# 04: Implicit ALS Matrix Factorization vs TruncatedSVD Baseline
This notebook compares Implicit Feedback ALS (with confidence scaling $C_{ui} = 1 + \alpha r_{ui}$) against the classical TruncatedSVD baseline.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np

# Robust root resolver whether run from notebooks/ or workspace root
_curr = Path('.').resolve()
PROJECT_ROOT = _curr if (_curr / 'src').exists() else (_curr.parent if (_curr.parent / 'src').exists() else (_curr.parents[1] if len(_curr.parents) > 1 and (_curr.parents[1] / 'src').exists() else _curr))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Switch working directory to project root so all relative paths ('data/...', 'artifacts/...') resolve seamlessly
os.chdir(PROJECT_ROOT)

from src.features.interaction_features import InteractionMatrixBuilder
from src.recommenders.als import ALSRecommender
from src.recommenders.svd import SVDRecommender


## 1. Load Matrix & Train ALS Model

In [ ]:
train_df = pd.read_parquet('data/processed/train_events.parquet')
id_builder = InteractionMatrixBuilder().fit_mappings(train_df)
csr_mat, _ = id_builder.build_matrix(train_df)

als = ALSRecommender(factors=64, regularization=0.05, iterations=20, alpha_confidence=40.0)
als.fit(csr_mat, id_builder)
print("ALS Model Fitted.")


## 2. Train TruncatedSVD Baseline

In [ ]:
svd = SVDRecommender(n_components=32)
svd.fit(csr_mat, id_builder)
var_sum = svd.svd.explained_variance_ratio_.sum() if svd.svd is not None and hasattr(svd.svd, "explained_variance_ratio_") else 0.0
print(f"TruncatedSVD Fitted. Explained Variance Sum: {var_sum:.4f}")


## 3. Compare Personalization Recommendations

In [ ]:
sample_user = list(id_builder.user_to_idx.keys())[0]
als_recs = als.recommend(sample_user, n=5)
svd_recs = svd.recommend(sample_user, n=5)

print(f"ALS Top-5 for Customer #{sample_user}: {als_recs}")
print(f"SVD Top-5 for Customer #{sample_user}: {svd_recs}")
